# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 5 (2, 3)
размер (2, 3) потому что сделали reshape такой формы, а 5 потому что это 3 элемент во 2 строке
2. 1 8
потому что покажет только первое максимальное значение, а его индекс - 1
3.  99
с шагом 3 получается [0, 3, 6, 9], а раз б не копия а, тогда меняем 0 на 99
4. 2 0.5
создается булев массив, True + True = 2, среднее 0.5
5. (2, 3) 0.25
создаем массив из 2 строк и 3 столбцов, первый элемент из списка от 0 до 1 поделенный на 5 равных частей = 0.25


In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [3]:
# База
# 1 Найдите позицию (argmax) самого дорогого заказа. Какая сумма, клиент и город
print(amount.max(), amount.argmax())

9999990.0 1001


In [4]:
amount[amount == amount.max()]

array([9999990., 9999990., 9999990.])

In [5]:
orders_max = []
for i in range(len(amount)):
    if amount[i] == amount.max():
        orders_max.append((i, cust[i], city[i], amount[i]))
print(orders_max)

[(1001, np.str_('TEST'), np.str_('Алматы'), np.float64(9999990.0)), (2268, np.str_('TEST'), np.str_('Алматы'), np.float64(9999990.0)), (2966, np.str_('TEST'), np.str_('Алматы'), np.float64(9999990.0))]


In [6]:
for i, cid, c, a in orders_max:
    print(f'индекс: {i}, id клиента: {cid}, город: {c}, сумма: {a:.0f}')

индекс: 1001, id клиента: TEST, город: Алматы, сумма: 9999990
индекс: 2268, id клиента: TEST, город: Алматы, сумма: 9999990
индекс: 2966, id клиента: TEST, город: Алматы, сумма: 9999990


Коментарии: Нашли самый большой заказ (9999990) и первое упоминание его индекса (1001). Но из прошлых заданий мы знаем что таких заказов несколько, поэтому для проверки и нахождения всех индексов пришлось использовать цикл for. Индексы: 1001, 2268, 2966. Сумма для всех трех заказов: 9999990, город у всех трех заказов - Алматы, клиент "TEST".

In [7]:
# 2 Выручка оплаченных заказов и сумма неоплаченных — через маску paid и ~paid
print(f'Выручка оплаченных заказов:', amount[paid].sum())
print(f'Сумма неоплаченных заказов:', amount[~paid].sum())

Выручка оплаченных заказов: 101909360.0
Сумма неоплаченных заказов: 7974000.0


In [8]:
# 3 Сколько заказов в Шымкенте и какая у них доля от всех заказов?
print(f'Заказов в Шымкенте: {len(city[city == 'Шымкент'])}, доля: {len(city[city == 'Шымкент']) / len(city) * 100:.2f}%')

Заказов в Шымкенте: 804, доля: 14.72%


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [9]:
# Стандарт
# 4 Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит? 
# Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
amount_after_discount = (amount[(paid) & (amount > 50_000)]) * 0.85
amount_diff = (amount[(paid) & (amount > 50_000)]).sum() - amount_after_discount.sum()
print(f'Магазин недополучит:', amount_diff)

Магазин недополучит: 5702422.5


In [10]:
# без TEST
print(len(paid[cust == 'TEST']))
print(paid[cust == 'TEST'])

3
[ True  True  True]


Комментарии: Все 3 заказа TEST имеют один статус - оплаченные, и, как нам ранее было известно, их значения больше 50 000, поэтому они учитывались при подсчете скидки.

In [11]:
all_paid_orders_without_test = amount[(paid) & (cust != 'TEST') & (amount > 50_000)]
discounted_paid_orders_without_test = all_paid_orders_without_test * 0.85

amount_diff_without_test = all_paid_orders_without_test.sum() - (discounted_paid_orders_without_test).sum()
print(f'Магазин недополучит:', amount_diff_without_test)

Магазин недополучит: 1202427.0


С учетом TEST, магазин недополучит 5 702 422, а без тестовых заказов - 1 202 427. Ответы отличаются в несколько раз, потому что три оплаченных тестовых заказа были очень дорогими (почти 10 млн), и после скидки разница между недополучкой с тестом и без должна быть 3 раза по 9999990 со учетом скидки 15% (3 * 9999990 * 0.15). Сделаем проверку:

In [12]:
print(f'Проверка:', 3 * 9999990 * 0.15 == amount_diff - amount_diff_without_test)

Проверка: True


In [13]:
# 5 Для каждого города — доля заказов, доставленных за 1–2 дня. Список городов: np.unique(city); цикл по 6 городам здесь можно.

Комменатрии: Раз в задании нет уточнения какие именно заказы считаются доставленными и нет пояснения как ведется учет в столбце `delivery_days`, тогда уточняю недостающую информацию у менеджера Дала Маркет.

А пока что, подразумеваю, что нас интерсует как быстро и вовремя доставляют заказы, подсчитываю абсолютно все заказы (отмененные и оплаченные), а для расчета долей использую данные, где `delivery_days` отражает факт совершения доставки и чей срок доставки составляет 1-2 дня. Результат для каждого города представлен в таблице ниже.

In [14]:
city_name = np.unique(city)
print(f'Города:', city_name)

Города: ['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']


In [15]:
for c in city_name:
    total = 0
    count = 0
    for c_1, d in zip(city, days):
        if c_1 == c:
            total += 1
        if c_1 == c and (d == 1 or d == 2):
            count += 1
    print(c, count, round(count / total * 100, 1),'%')

Актобе 247 62.2 %
Алматы 1412 64.5 %
Астана 883 63.5 %
Атырау 94 64.4 %
Караганда 365 67.8 %
Шымкент 240 29.9 %


In [16]:
# 6 Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
# avg = amount.mean()
# print("Средний чек:", round(avg))

Комменатрии: Возможно, правильней считать средний чек среди оплаченных заказов, а также стоит исключить оплаченные тестовые заказы из расчета.

In [17]:
avg = amount[paid & (cust != 'TEST')].mean()
print("Средний чек оплаченных заказов:", round(avg))

Средний чек оплаченных заказов: 14672


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [18]:
# Челлендж
# 7 Сравните скорость: сумма оплаченных заказов циклом for и через маску. Во сколько раз быстрее?

In [19]:
import time

In [20]:
t = time.time()
s = 0
for a, p in zip(amount, paid):
    if p:
        s += a
t_loop = time.time() - t

In [21]:
t = time.time()
s2 = amount[paid].sum()
t_np = time.time() - t

In [22]:
print(f"Цикл for: {t_loop:.4f} cек.; NumPy: {t_np:.4f} cек., быстрее в {t_loop / t_np:.0f} раз")

Цикл for: 0.0019 cек.; NumPy: 0.0002 cек., быстрее в 10 раз


In [23]:
# 8 Смоделируйте A/B-тест. np.random.seed(42). Группа A: 1 000 клиентов, вероятность покупки 5%; 
# группа B: 1 000 клиентов, 6% (покупка = np.random.rand(1000) < 0.05). Сколько покупок в каждой группе? 
# Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [24]:
# без решения
# np.random.seed(42)

# a_prob = 0.05
#b_prob = 0.06

In [25]:
#a_buy_count = 0
#for a in range(100):
#    for a1 in np.random.rand(1000):
#        if a1 < a_prob:
#            a_buy_count += 1
#print(a_buy_count)

In [26]:
#b_buy_count = 0
#for b in range(100):
#    for b1 in np.random.rand(1000):
#        if b1 < b_prob:
#            b_buy_count += 1
#print(b_buy_count)